In [ ]:
5

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

visited = set()

def dfs_crawl(url, base_url, depth=0, max_depth=3):

    if depth > max_depth or url in visited:
        return

    print("Visiting:", url)

    visited.add(url)

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.find_all("a", href=True)

    for link in links:

        absolute_url = urljoin(url, link["href"])

        if urlparse(absolute_url).netloc == urlparse(base_url).netloc:
            dfs_crawl(absolute_url, base_url, depth + 1, max_depth)

In [ ]:
dfs_crawl("https://www.tensorflow.org/tutorials",
          "https://www.tensorflow.org",
          max_depth=2)

# Simple DFS - without filtering

In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

visited = set()

def dfs_crawl(url, base_url, depth=0, max_depth=3): #✅✅ DEPTH

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        response = requests.get(url, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Extract text
        text = soup.get_text(separator=" ", strip=True)

        # Print sample of text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        links = soup.find_all("a", href=True)

        # Only first 3 links
        links = links[:3]  #✅✅ BRANCHING FACTOR

        for link in links:

            absolute_url = urljoin(url, link["href"])

            if urlparse(absolute_url).netloc == urlparse(base_url).netloc:
                dfs_crawl(absolute_url, base_url, depth + 1, max_depth)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org",
    max_depth=2
)


Visiting: https://www.tensorflow.org/tutorials
Sample Text: Tutorials  |  TensorFlow Core Skip to main content Install Learn Introduction New to TensorFlow? Tutorials Learn how to use TensorFlow with end-to-end examples Guide Learn framework concepts and components Learn ML Educational resources to master your path with TensorFlow API TensorFlow (v2.16.1) Ve


  Visiting: https://www.tensorflow.org/tutorials#main-content
  Sample Text: Tutorials  |  TensorFlow Core Skip to main content Install Learn Introduction New to TensorFlow? Tutorials Learn how to use TensorFlow with end-to-end examples Guide Learn framework concepts and components Learn ML Educational resources to master your path with TensorFlow API TensorFlow (v2.16.1) Ve


    Visiting: https://www.tensorflow.org/
    Sample Text: TensorFlow Skip to main content Install Learn Introduction New to TensorFlow? Tutorials Learn how to use TensorFlow with end-to-end examples Guide Learn framework concepts and components Learn ML E

# DFS paths with URL Filter (#-removed)

In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse

visited = set()

def normalize_and_filter_url(base_url, current_url, href):

    # Convert to absolute URL
    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)

    # Remove fragment (#...)
    parsed = parsed._replace(fragment="")

    # Remove query params (?hl=en)
    parsed = parsed._replace(query="")

    # Normalize path (remove trailing slash)
    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)

    clean_url = urlunparse(parsed)

    # Same domain check
    if urlparse(clean_url).netloc != urlparse(base_url).netloc:
        return None

    # Restrict to tutorials path
    if not clean_url.startswith(base_url.rstrip("/")):
        return None

    # Skip non-html resources
    if clean_url.endswith((".pdf", ".jpg", ".png", ".zip")):
        return None

    return clean_url


def dfs_crawl(url, base_url, depth=0, max_depth=3):

    # Normalize current URL as well (IMPORTANT)
    url = url.rstrip("/")

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()

        # Extract text
        text = soup.get_text(separator=" ", strip=True)

        # Print sample text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        raw_links = soup.find_all("a", href=True)

        clean_links = []
        seen_links = set()

        for link in raw_links:

            clean_url = normalize_and_filter_url(base_url, url, link["href"])

            # Skip invalid
            if not clean_url:
                continue

            # Skip duplicates (VERY IMPORTANT)
            if clean_url in seen_links:
                continue

            # Skip already visited
            if clean_url in visited:
                continue

            # Add to lists
            seen_links.add(clean_url)
            clean_links.append(clean_url)

            # Limit AFTER filtering
            if len(clean_links) == 3:
                break

        # Debug: show children
        for cl in clean_links:
            print(f"{'  '*depth}→ Next: {cl}")

        # DFS recursion
        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1, max_depth)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2
)


Visiting: https://www.tensorflow.org/tutorials
Sample Text: Tutorials  |  TensorFlow Core Skip to main content / English Español – América Latina Français Indonesia Italiano Polski Português – Brasil Tiếng Việt Türkçe Русский עברית العربيّة فارسی हिंदी বাংলা ภาษาไทย 中文 – 简体 中文 – 繁體 日本語 한국어 GitHub Sign in TensorFlow Core An open source machine learning libr

→ Next: https://www.tensorflow.org/tutorials/quickstart/beginner
→ Next: https://www.tensorflow.org/tutorials/keras/classification
→ Next: https://www.tensorflow.org/tutorials/load_data/csv

  Visiting: https://www.tensorflow.org/tutorials/quickstart/beginner
  Sample Text: TensorFlow 2 quickstart for beginners  |  TensorFlow Core Skip to main content / English Español – América Latina Français Indonesia Italiano Polski Português – Brasil Tiếng Việt Türkçe Русский עברית العربيّة فارسی हिंदी বাংলা ภาษาไทย 中文 – 简体 日本語 한국어 GitHub Sign in TensorFlow Core TensorFlow Learn

  → Next: https://www.tensorflow.org/tutorials/keras
  → Nex

# Header & Footer Removed

In [14]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse

visited = set()

def normalize_and_filter_url(base_url, current_url, href):

    # Convert to absolute URL
    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)

    # Remove fragment (#...)
    parsed = parsed._replace(fragment="")

    # Remove query params (?hl=en)
    parsed = parsed._replace(query="")

    # Normalize path (remove trailing slash)
    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)

    clean_url = urlunparse(parsed)

    # Same domain check
    if urlparse(clean_url).netloc != urlparse(base_url).netloc:
        return None

    # Restrict to tutorials path
    if not clean_url.startswith(base_url.rstrip("/")):
        return None

    # Skip non-html resources
    if clean_url.endswith((".pdf", ".jpg", ".png", ".zip")):
        return None

    return clean_url


def dfs_crawl(url, base_url, depth=0, max_depth=3):

    # Normalize current URL as well (IMPORTANT)
    url = url.rstrip("/")

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags (stronger cleaning)
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        # Try extracting MAIN content only
        main = soup.find("main")

        if main:
            text = main.get_text(" ", strip=True)
        else:
            text = soup.get_text(" ", strip=True)

        # Clean extra whitespace
        text = " ".join(text.split())

        # Print sample text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        raw_links = soup.find_all("a", href=True)

        clean_links = []
        seen_links = set()

        for link in raw_links:

            clean_url = normalize_and_filter_url(base_url, url, link["href"])

            # Skip invalid
            if not clean_url:
                continue

            # Skip duplicates (VERY IMPORTANT)
            if clean_url in seen_links:
                continue

            # Skip already visited
            if clean_url in visited:
                continue

            # Add to lists
            seen_links.add(clean_url)
            clean_links.append(clean_url)

            # Limit AFTER filtering
            if len(clean_links) == 3:
                break

        # Debug: show children
        for cl in clean_links:
            print(f"{'  '*depth}→ Next: {cl}")

        # DFS recursion
        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1, max_depth)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2
)


Visiting: https://www.tensorflow.org/tutorials
Sample Text: TensorFlow Learn TensorFlow Core Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll

→ Next: https://www.tensorflow.org/tutorials/quickstart/beginner
→ Next: https://www.tensorflow.org/tutorials/keras/classification
→ Next: https://www.tensorflow.org/tutorials/load_data/csv

  Visiting: https://www.tensorflow.org/tutorials/quickstart/beginner
  Sample Text: TensorFlow Learn TensorFlow Core TensorFlow 2 quickstart for beginners Stay organized with collections Save and categorize content based on your preferences. View on TensorFlow.org Run in Google Colab View source on GitHub Download notebook This short introduction uses Keras to: Load a prebuilt data

  → Next: https://www.tensorflow.org/tutorials/keras
  → Nex

# URL FILTERING - FilterChain, DomainFilter, URLPatternFilter, ContentTypeFilter

In [1]:
5

5

Here - dfs_crawl finds a raw link\
       ↓\
normalize_and_filter_url() cleans it\
       ↓\
FilterChain.allow() runs it through:\
   DomainFilter → URLPatternFilter → ContentTypeFilter\
       ↓\
Only URLs passing all 3 filters are followed

In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib
import filters
importlib.reload(filters)
from filters import FilterChain, DomainFilter, URLPatternFilter, ContentTypeFilter

visited = set()

filter_chain = FilterChain([
    DomainFilter("https://www.tensorflow.org/tutorials"),
    URLPatternFilter("https://www.tensorflow.org/tutorials"),
    ContentTypeFilter()
],
debug=True #to check the link is passing through the filters or not
)

def normalize_and_filter_url(base_url, current_url, href):

    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)

    parsed = parsed._replace(fragment="")
    parsed = parsed._replace(query="")

    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)

    clean_url = urlunparse(parsed)

    # 🔥 Apply filter chain
    if not filter_chain.allow(clean_url):
        return None

    return clean_url


def dfs_crawl(url, base_url, depth=0, max_depth=3, branching_factor=3):

    # Normalize current URL as well (IMPORTANT)
    url = url.rstrip("/")

    if depth > max_depth or url in visited:
        return

    print(f"\n{'  '*depth}Visiting: {url}")

    visited.add(url)

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unwanted tags (stronger cleaning)
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        # Try extracting MAIN content only
        main = soup.find("main")

        if main:
            text = main.get_text(" ", strip=True)
        else:
            text = soup.get_text(" ", strip=True)

        # Clean extra whitespace
        text = " ".join(text.split())

        # Print sample text
        sample = text[:300]
        print(f"{'  '*depth}Sample Text: {sample}\n")

        # Extract links
        raw_links = soup.find_all("a", href=True)

        clean_links = []
        seen_links = set()

        for link in raw_links:

            clean_url = normalize_and_filter_url(base_url, url, link["href"])

            # Skip invalid
            if not clean_url:
                continue

            # Skip duplicates (VERY IMPORTANT)
            if clean_url in seen_links:
                continue

            # Skip already visited
            if clean_url in visited:
                continue

            # Add to lists
            seen_links.add(clean_url)
            clean_links.append(clean_url)

            # Limit AFTER filtering
            if len(clean_links) >= branching_factor:
                break

        # Debug: show children
        for cl in clean_links:
            print(f"{'  '*depth}→ Next: {cl}")

        # DFS recursion
        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1, max_depth, branching_factor)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2,
    branching_factor=2
)


Visiting: https://www.tensorflow.org/tutorials
Sample Text: TensorFlow Learn TensorFlow Core Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll

[Filter: DomainFilter] https://www.tensorflow.org/tutorials → True
[Filter: URLPatternFilter] https://www.tensorflow.org/tutorials → True
[Filter: ContentTypeFilter] https://www.tensorflow.org/tutorials → True
[Filter: DomainFilter] https://www.tensorflow.org → True
[Filter: URLPatternFilter] https://www.tensorflow.org → False
[Filter: DomainFilter] https://github.com/tensorflow → False
[Filter: DomainFilter] https://www.tensorflow.org/tutorials → True
[Filter: URLPatternFilter] https://www.tensorflow.org/tutorials → True
[Filter: ContentTypeFilter] https://www.tensorflow.org/tutorials → True
[Filter: DomainFilter] https://www.te

# Saving the explored DFS structure in JSON - visualizing in visualize_tree.html

In [8]:
import requests
import json
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib
import filters
importlib.reload(filters)
from filters import FilterChain, DomainFilter, URLPatternFilter, ContentTypeFilter

visited = set()

filter_chain = FilterChain([
    DomainFilter("https://www.tensorflow.org/tutorials"),
    # URLPatternFilter("https://www.tensorflow.org/tutorials"),
    ContentTypeFilter()
], debug=False)


def normalize_and_filter_url(base_url, current_url, href):
    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)
    parsed = parsed._replace(fragment="")
    parsed = parsed._replace(query="")
    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)
    clean_url = urlunparse(parsed)
    if not filter_chain.allow(clean_url):
        return None
    return clean_url


def dfs_crawl(url, base_url, depth=0, max_depth=2, max_links=3):
    url = url.rstrip("/")

    if depth > max_depth or url in visited:
        return None                          # ← return None if skipped

    print(f"{'  ' * depth}[d{depth}] Visiting: {url}")
    visited.add(url)

    # ── build the node that will go into the JSON tree ──
    node = {
        "url": url,
        "depth": depth,
        "status": "ok",
        "children": []
    }

    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=5)

        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        main = soup.find("main")
        text = main.get_text(" ", strip=True) if main else soup.get_text(" ", strip=True)
        text = " ".join(text.split())

        node["sample"] = text[:200]          # store a snippet for the tooltip

        raw_links = soup.find_all("a", href=True)
        clean_links = []
        seen_links = set()

        for link in raw_links:
            clean_url = normalize_and_filter_url(base_url, url, link["href"])
            if not clean_url:
                continue
            if clean_url in seen_links or clean_url in visited:
                continue
            seen_links.add(clean_url)
            clean_links.append(clean_url)
            if len(clean_links) == max_links:   # ← configurable now
                break

        for link in clean_links:
            child_node = dfs_crawl(link, base_url, depth + 1, max_depth, max_links)
            if child_node:
                node["children"].append(child_node)

    except Exception as e:
        node["status"] = "error"
        node["error"] = str(e)
        print(f"{'  ' * depth}  Error: {e}")

    return node


# ── Run & save ──────────────────────────────────────────
tree = dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2,
    max_links=3
)

with open("crawl_tree.json", "w") as f:
    json.dump(tree, f, indent=2)

print("\n✓ Tree saved to crawl_tree.json")
print(f"  Total pages visited: {len(visited)}")

[d0] Visiting: https://www.tensorflow.org/tutorials
  [d1] Visiting: https://www.tensorflow.org
    [d2] Visiting: https://www.tensorflow.org/install
    [d2] Visiting: https://www.tensorflow.org/js
    [d2] Visiting: https://www.tensorflow.org/guide/data
  [d1] Visiting: https://www.tensorflow.org/learn
    [d2] Visiting: https://www.tensorflow.org/tfx
    [d2] Visiting: https://www.tensorflow.org/datasets
    [d2] Visiting: https://www.tensorflow.org/guide/keras/preprocessing_layers
  [d1] Visiting: https://www.tensorflow.org/guide/keras/overview
    [d2] Visiting: https://www.tensorflow.org/guide
    [d2] Visiting: https://www.tensorflow.org/guide/core
    [d2] Visiting: https://www.tensorflow.org/guide/keras/sequential_model

✓ Tree saved to crawl_tree.json
  Total pages visited: 13


# Adding new parameters - max_pages, branching_factor

In [13]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib
import filters
importlib.reload(filters)
from filters import FilterChain, DomainFilter, URLPatternFilter, ContentTypeFilter

visited = set()

filter_chain = FilterChain([
    DomainFilter("https://www.tensorflow.org/tutorials"),
    # URLPatternFilter("https://www.tensorflow.org/tutorials"),
    ContentTypeFilter()
], debug=False)

def normalize_and_filter_url(base_url, current_url, href):
    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)
    parsed = parsed._replace(fragment="")
    parsed = parsed._replace(query="")
    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)
    clean_url = urlunparse(parsed)
    if not filter_chain.allow(clean_url):
        return None
    return clean_url


def dfs_crawl(url, base_url, depth=0,
              max_depth=None,           # None = unlimited
              branching_factor=None,    # None = unlimited  
              max_pages=None,           # None = unlimited
              page_count=None):

    if page_count is None:
        page_count = [0]

    url = url.rstrip("/")

    # depth guard — only applies if max_depth is set
    if max_depth is not None and depth > max_depth:
        return

    if url in visited:
        return

    # page budget guard — only applies if max_pages is set
    if max_pages is not None and page_count[0] >= max_pages:
        print(f"\n[max_pages={max_pages} reached — stopping]")
        return

    print(f"\n{'  '*depth}Visiting [{page_count[0]+1}]: {url}")

    visited.add(url)
    page_count[0] += 1

    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=5)

        # ── Content-Type check on actual response (not HEAD) ──
        content_type = response.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            print(f"{'  '*depth}Skipping non-HTML: {content_type}")
            return

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        main = soup.find("main")
        text = main.get_text(" ", strip=True) if main else soup.get_text(" ", strip=True)
        text = " ".join(text.split())

        print(f"{'  '*depth}Sample Text: {text[:300]}\n")

        raw_links = soup.find_all("a", href=True)
        clean_links = []
        seen_links = set()

        for link in raw_links:
            clean_url = normalize_and_filter_url(base_url, url, link["href"])
            if not clean_url:
                continue
            if clean_url in seen_links or clean_url in visited:
                continue
            seen_links.add(clean_url)
            clean_links.append(clean_url)
            # branching guard — only applies if branching_factor is set
            if branching_factor is not None and len(clean_links) >= branching_factor:
                break

        for cl in clean_links:
            print(f"{'  '*depth}-> Next: {cl}")

        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1,
                      max_depth, branching_factor, max_pages, page_count)

    except Exception as e:
        print("Error:", e)


# Start crawling
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2,
    branching_factor=3,
    max_pages=5                                        
)


Visiting [1]: https://www.tensorflow.org/tutorials
Sample Text: TensorFlow Learn TensorFlow Core Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll

-> Next: https://www.tensorflow.org
-> Next: https://www.tensorflow.org/learn
-> Next: https://www.tensorflow.org/guide/keras/overview

  Visiting [2]: https://www.tensorflow.org
  Sample Text: TensorFlow Stay organized with collections Save and categorize content based on your preferences. An end-to-end platform for machine learning Install TensorFlow Get started with TensorFlow TensorFlow makes it easy to create ML models that can run in any environment. Learn how to use the intuitive AP

  -> Next: https://www.tensorflow.org/install
  -> Next: https://www.tensorflow.org/js
  -> Next: https://www.tensorflow.org/guide/data



# Added score_threshold, exclude_social_media and word_count_threshold

In [16]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib
import filters
importlib.reload(filters)
from filters import (FilterChain, DomainFilter, URLPatternFilter,
                     ContentTypeFilter, ExternalLinkFilter, SocialMediaFilter)


visited = set()

filter_chain = FilterChain([
    DomainFilter("https://www.tensorflow.org/tutorials"),
    URLPatternFilter("https://www.tensorflow.org/tutorials"),
    ExternalLinkFilter("https://www.tensorflow.org/tutorials"),
    SocialMediaFilter(),
    ContentTypeFilter(),
], debug=False)


# ── Link scorer ────────────────────────────────────────────────────────────────
#
#   Scores a candidate link 0.0–1.0 based on signals from the URL and
#   anchor text. No ML — pure heuristics, fast, zero extra requests.
#
#   Signals:
#     - URL path shares keywords with base URL path      → up to 0.5
#     - inverse depth penalty (shallower = better)       → up to 0.2
#     - anchor text is present and non-trivial           → up to 0.3
#
#   score_threshold=None  → scoring disabled, all links accepted
#   score_threshold=0.3   → only links scoring ≥ 0.3 are followed
#
def score_link(href, anchor_text, base_url):
    score = 0.0

    base_keywords = set(urlparse(base_url).path.strip("/").split("/"))
    base_keywords.discard("")

    link_parts = set(urlparse(href).path.strip("/").split("/"))
    link_parts.discard("")

    # signal 1: keyword overlap with base path
    overlap = len(base_keywords & link_parts)
    score += min(overlap / max(len(base_keywords), 1), 1.0) * 0.5

    # signal 2: depth penalty — each extra segment costs 0.05, capped at 0.2
    depth_penalty = min(len(link_parts) * 0.05, 0.2)
    score += (0.2 - depth_penalty)

    # signal 3: non-trivial anchor text
    if len(anchor_text.strip()) > 3:
        score += 0.3

    return round(min(score, 1.0), 3)


def normalize_and_filter_url(base_url, current_url, href):
    absolute_url = urljoin(current_url, href)
    parsed = urlparse(absolute_url)
    parsed = parsed._replace(fragment="")
    parsed = parsed._replace(query="")
    normalized_path = parsed.path.rstrip("/")
    parsed = parsed._replace(path=normalized_path)
    clean_url = urlunparse(parsed)

    # Stage 1: URL-only checks (domain, pattern, external, social, extension)
    if not filter_chain.allow(clean_url):
        return None

    return clean_url


def dfs_crawl(url, base_url, depth=0,
              max_depth=None,
              branching_factor=None,
              max_pages=None,
              score_threshold=None,       # None = disabled, e.g. 0.3
              word_count_threshold=None,  # None = disabled, e.g. 50
              page_count=None):

    if page_count is None:
        page_count = [0]

    url = url.rstrip("/")

    if max_depth is not None and depth > max_depth:
        return

    if url in visited:
        return

    if max_pages is not None and page_count[0] >= max_pages:
        print(f"\n[max_pages={max_pages} reached — stopping]")
        return

    print(f"\n{'  '*depth}Visiting [{page_count[0]+1}]: {url}")

    visited.add(url)
    page_count[0] += 1

    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=5)

        # Stage 2: Content-Type from real response — no extra request
        if not filter_chain.allow_response(url, response):
            print(f"{'  '*depth}Skipped — Content-Type blocked: "
                  f"{response.headers.get('Content-Type', 'unknown')}")
            return

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        main = soup.find("main")
        text = main.get_text(" ", strip=True) if main else soup.get_text(" ", strip=True)
        text = " ".join(text.split())

        # ── word count threshold ───────────────────────────────────────────────
        # Runs on THIS page's extracted text after cleaning.
        # Skips thin pages: stubs, redirect pages, empty index pages.
        # Does NOT prevent recursion from parent — just skips this node's content.
        if word_count_threshold is not None:
            word_count = len(text.split())
            if word_count < word_count_threshold:
                print(f"{'  '*depth}Skipped — only {word_count} words "
                      f"(threshold={word_count_threshold})")
                return

        print(f"{'  '*depth}Sample Text: {text[:300]}\n")

        # ── link collection with optional scoring ──────────────────────────────
        raw_links = soup.find_all("a", href=True)
        clean_links = []
        seen_links = set()

        for link in raw_links:
            clean_url = normalize_and_filter_url(base_url, url, link["href"])
            if not clean_url:
                continue
            if clean_url in seen_links or clean_url in visited:
                continue

            # ── score threshold ────────────────────────────────────────────────
            # Heuristic score from URL structure + anchor text.
            # Filters OUT low-relevance links before they waste a crawl slot.
            # score_threshold=None disables scoring — all passing links accepted.
            if score_threshold is not None:
                anchor_text = link.get_text()
                link_score = score_link(clean_url, anchor_text, base_url)
                if link_score < score_threshold:
                    print(f"{'  '*depth}  [score={link_score} < {score_threshold}]"
                          f" skipped: {clean_url}")
                    continue

            seen_links.add(clean_url)
            clean_links.append(clean_url)

            if branching_factor is not None and len(clean_links) >= branching_factor:
                break

        for cl in clean_links:
            print(f"{'  '*depth}-> Next: {cl}")

        for link in clean_links:
            dfs_crawl(link, base_url, depth + 1,
                      max_depth, branching_factor, max_pages,
                      score_threshold, word_count_threshold, page_count)

    except Exception as e:
        print("Error:", e)


# ── Example calls ──────────────────────────────────────────────────────────────

# All limits off
# dfs_crawl(
#     "https://www.tensorflow.org/tutorials",
#     "https://www.tensorflow.org/tutorials",
# )

# Fully controlled
dfs_crawl(
    "https://www.tensorflow.org/tutorials",
    "https://www.tensorflow.org/tutorials",
    max_depth=2,
    branching_factor=3,
    max_pages=5,
    score_threshold=0.3,
    word_count_threshold=50,
)


Visiting [1]: https://www.tensorflow.org/tutorials
Sample Text: TensorFlow Learn TensorFlow Core Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll

-> Next: https://www.tensorflow.org/tutorials/quickstart/beginner
-> Next: https://www.tensorflow.org/tutorials/keras/classification
-> Next: https://www.tensorflow.org/tutorials/load_data/csv

  Visiting [2]: https://www.tensorflow.org/tutorials/quickstart/beginner
  Sample Text: TensorFlow Learn TensorFlow Core TensorFlow 2 quickstart for beginners Stay organized with collections Save and categorize content based on your preferences. View on TensorFlow.org Run in Google Colab View source on GitHub Download notebook This short introduction uses Keras to: Load a prebuilt data

  -> Next: https://www.tensorflow.org/tutorials/k

# ALL from 1 Function init main

In [17]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib
import filters
importlib.reload(filters)
from filters import (FilterChain, DomainFilter, URLPatternFilter,
                     ContentTypeFilter, ExternalLinkFilter, SocialMediaFilter)


def crawl(
    start_url,

    # ── crawl controls ─────────────────────────────────────────
    max_depth=None,             # None = unlimited depth
    branching_factor=None,      # None = follow all links per page
    max_pages=None,             # None = no page budget

    # ── content controls ───────────────────────────────────────
    score_threshold=None,       # None = disabled  |  e.g. 0.3
    word_count_threshold=None,  # None = disabled  |  e.g. 50

    # ── filter toggles ─────────────────────────────────────────
    url_pattern=None,           # None = no path restriction  |  e.g. "https://example.com/docs"
    exclude_external=False,     # True = stay on start_url's domain only
    exclude_social_media=False, # True = drop twitter/linkedin/etc.
    extra_social_domains=None,  # additional domains to block e.g. ["medium.com"]

    # ── debug ──────────────────────────────────────────────────
    debug=False,
):
    """
    Single entry point for the DFS crawler.

    All filter configuration, crawl limits, and content thresholds
    are set here. Nothing needs to be touched elsewhere.

    Parameters
    ----------
    start_url             : str   — the URL to start crawling from
    max_depth             : int   — how deep to recurse (None = unlimited)
    branching_factor      : int   — max links to follow per page (None = all)
    max_pages             : int   — total page budget across whole crawl (None = unlimited)
    score_threshold       : float — min relevance score for a link to be followed (None = off)
    word_count_threshold  : int   — skip pages with fewer words than this (None = off)
    url_pattern           : str   — only follow URLs starting with this prefix (None = off)
    exclude_external      : bool  — block links leaving start_url's domain
    exclude_social_media  : bool  — block links to social media domains
    extra_social_domains  : list  — additional domains to treat as social media
    debug                 : bool  — print filter decisions per URL
    """

    # ── build filter chain from params ────────────────────────────────────────
    active_filters = [
        DomainFilter(start_url),                        # always on — stay on same domain
    ]

    if url_pattern is not None:
        active_filters.append(URLPatternFilter(url_pattern))

    if exclude_external:
        active_filters.append(ExternalLinkFilter(start_url))

    if exclude_social_media:
        active_filters.append(SocialMediaFilter(extra_domains=extra_social_domains))

    active_filters.append(ContentTypeFilter())          # always last — stage 1+2

    filter_chain = FilterChain(active_filters, debug=debug)

    # ── crawl state — local to this crawl() call ──────────────────────────────
    visited    = set()
    page_count = [0]

    # ── helpers — closed over filter_chain and base_url ───────────────────────

    def normalize_and_filter_url(current_url, href):
        absolute_url = urljoin(current_url, href)
        parsed = urlparse(absolute_url)
        parsed = parsed._replace(fragment="")
        parsed = parsed._replace(query="")
        parsed = parsed._replace(path=parsed.path.rstrip("/"))
        clean_url = urlunparse(parsed)
        return clean_url if filter_chain.allow(clean_url) else None

    def score_link(href, anchor_text):
        """
        Heuristic relevance score 0.0–1.0.

        Signal 1 (0–0.5): URL path keywords overlap with start_url path.
        Signal 2 (0–0.2): Shallow depth is preferred (less penalty).
        Signal 3 (0–0.3): Anchor text is non-trivial (len > 3).
        """
        score = 0.0
        base_kw   = set(urlparse(start_url).path.strip("/").split("/")) - {""}
        link_parts = set(urlparse(href).path.strip("/").split("/"))     - {""}

        overlap = len(base_kw & link_parts)
        score += min(overlap / max(len(base_kw), 1), 1.0) * 0.5
        score += 0.2 - min(len(link_parts) * 0.05, 0.2)
        if len(anchor_text.strip()) > 3:
            score += 0.3

        return round(min(score, 1.0), 3)

    # ── private recursive worker ───────────────────────────────────────────────

    def _dfs_crawl(url, depth):

        url = url.rstrip("/")

        if max_depth is not None and depth > max_depth:
            return
        if url in visited:
            return
        if max_pages is not None and page_count[0] >= max_pages:
            print(f"[max_pages={max_pages} reached — stopping]")
            return

        indent = "  " * depth
        page_num = f"[{page_count[0]+1}" + (f"/{max_pages}]" if max_pages else "]")
        print(f"\n{indent}Visiting {page_num}: {url}")

        visited.add(url)
        page_count[0] += 1

        try:
            response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)

            # stage 2 content-type check (uses actual response, no extra request)
            if not filter_chain.allow_response(url, response):
                print(f"{indent}Skipped — Content-Type blocked: "
                      f"{response.headers.get('Content-Type', 'unknown')}")
                return

            soup = BeautifulSoup(response.text, "html.parser")

            for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
                tag.decompose()

            main = soup.find("main")
            text = main.get_text(" ", strip=True) if main else soup.get_text(" ", strip=True)
            text = " ".join(text.split())

            # word count gate — skip thin/stub pages
            if word_count_threshold is not None:
                wc = len(text.split())
                if wc < word_count_threshold:
                    print(f"{indent}Skipped — {wc} words < threshold({word_count_threshold})")
                    return

            print(f"{indent}Text ({len(text.split())} words): {text[:300]}\n")

            # collect, score, and limit candidate links
            clean_links = []
            seen_links  = set()

            for link in soup.find_all("a", href=True):
                clean_url = normalize_and_filter_url(url, link["href"])
                if not clean_url:
                    continue
                if clean_url in seen_links or clean_url in visited:
                    continue

                if score_threshold is not None:
                    s = score_link(clean_url, link.get_text())
                    if s < score_threshold:
                        print(f"{indent}  [score={s} < {score_threshold}] dropped: {clean_url}")
                        continue

                seen_links.add(clean_url)
                clean_links.append(clean_url)

                if branching_factor is not None and len(clean_links) >= branching_factor:
                    break

            for cl in clean_links:
                print(f"{indent}-> {cl}")

            for cl in clean_links:
                _dfs_crawl(cl, depth + 1)

        except Exception as e:
            print(f"{indent}Error: {e}")

    # ── kick off ───────────────────────────────────────────────────────────────
    _dfs_crawl(start_url, depth=0)
    print(f"\nCrawl complete — {page_count[0]} pages visited.")


# ── Usage ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # minimal — just a start URL, everything else defaults to unlimited/off
    # crawl("https://www.tensorflow.org/tutorials")

    # fully controlled
    crawl(
        start_url            = "https://www.tensorflow.org/tutorials",
        max_depth            = 2,
        branching_factor     = 3,
        max_pages            = 5,
        score_threshold      = 0.3,
        word_count_threshold = 50,
        url_pattern          = "https://www.tensorflow.org/tutorials",
        exclude_external     = True,
        exclude_social_media = True,
        extra_social_domains = ["medium.com"],
        debug                = False,
    )


Visiting [1/5]: https://www.tensorflow.org/tutorials
Text (516 words): TensorFlow Learn TensorFlow Core Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll

-> https://www.tensorflow.org/tutorials/quickstart/beginner
-> https://www.tensorflow.org/tutorials/keras/classification
-> https://www.tensorflow.org/tutorials/load_data/csv

  Visiting [2/5]: https://www.tensorflow.org/tutorials/quickstart/beginner
  Text (2403 words): TensorFlow Learn TensorFlow Core TensorFlow 2 quickstart for beginners Stay organized with collections Save and categorize content based on your preferences. View on TensorFlow.org Run in Google Colab View source on GitHub Download notebook This short introduction uses Keras to: Load a prebuilt data

  -> https://www.tensorflow.org/tutorials/keras
  ->

In [ ]:
from markdownify import markdownify as md
md('<b>Yay</b> <a href="http://github.com">GitHub</a>')

'**Yay** [GitHub](http://github.com)'

# Added Different Output types

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib

import filters
importlib.reload(filters)   
from filters import (FilterChain, DomainFilter, URLPatternFilter,
                     ContentTypeFilter, ExternalLinkFilter, SocialMediaFilter)

import outputs
importlib.reload(outputs)
from outputs import PageResult, OutputManager


def crawl(
    start_url,

    # ── output ─────────────────────────────────────────────────
    output_formats=None,        # list of str names or pre-configured BaseOutput instances
                                # e.g. ["markdown", "metadata"]
                                # e.g. [MarkdownOutput(strip_links=True), MetadataOutput()]
                                # e.g. ["json"]  → all formats, packed into to_json()
                                # None / []      → crawl only, no content stored

    # ── crawl controls ─────────────────────────────────────────
    max_depth=None,             # None = unlimited
    branching_factor=None,      # None = follow all links per page
    max_pages=None,             # None = no budget

    # ── content controls ───────────────────────────────────────
    score_threshold=None,       # None = off  |  e.g. 0.3
    word_count_threshold=None,  # None = off  |  e.g. 50

    # ── filter toggles ─────────────────────────────────────────
    url_pattern=None,           # None = off  |  e.g. "https://example.com/docs"
    exclude_external=False,
    exclude_social_media=False,
    extra_social_domains=None,

    # ── debug ──────────────────────────────────────────────────
    debug=False,
):
    """
    Single entry point for the DFS crawler.

    Parameters
    ----------
    start_url             : str
    output_formats        : list[str | BaseOutput]  (None = crawl only)
    max_depth             : int    (None = unlimited)
    branching_factor      : int    (None = all links)
    max_pages             : int    (None = unlimited)
    score_threshold       : float  (None = off)
    word_count_threshold  : int    (None = off)
    url_pattern           : str    (None = off)
    exclude_external      : bool
    exclude_social_media  : bool
    extra_social_domains  : list[str]
    debug                 : bool

    Returns
    -------
    list[PageResult]
    """

    # ── output manager ────────────────────────────────────────────────────────
    manager = OutputManager(output_formats or [])

    # ── filter chain ─────────────────────────────────────────────────────────
    active_filters = [DomainFilter(start_url)]

    if url_pattern is not None:
        active_filters.append(URLPatternFilter(url_pattern))
    if exclude_external:
        active_filters.append(ExternalLinkFilter(start_url))
    if exclude_social_media:
        active_filters.append(SocialMediaFilter(extra_domains=extra_social_domains))

    active_filters.append(ContentTypeFilter())

    filter_chain = FilterChain(active_filters, debug=debug)

    # ── crawl state ───────────────────────────────────────────────────────────
    visited    = set()
    page_count = [0]
    results    = []

    # ── helpers ───────────────────────────────────────────────────────────────

    def normalize_and_filter_url(current_url, href):
        absolute_url = urljoin(current_url, href)
        parsed       = urlparse(absolute_url)
        parsed       = parsed._replace(fragment="", query="")
        parsed       = parsed._replace(path=parsed.path.rstrip("/"))
        clean_url    = urlunparse(parsed)
        return clean_url if filter_chain.allow(clean_url) else None

    def score_link(href, anchor_text):
        score      = 0.0
        base_kw    = set(urlparse(start_url).path.strip("/").split("/")) - {""}
        link_parts = set(urlparse(href).path.strip("/").split("/"))      - {""}
        overlap    = len(base_kw & link_parts)
        score     += min(overlap / max(len(base_kw), 1), 1.0) * 0.5
        score     += 0.2 - min(len(link_parts) * 0.05, 0.2)
        if len(anchor_text.strip()) > 3:
            score += 0.3
        return round(min(score, 1.0), 3)

    # ── recursive worker ──────────────────────────────────────────────────────

    def _dfs_crawl(url, depth):

        url = url.rstrip("/")

        if max_depth is not None and depth > max_depth:
            return
        if url in visited:
            return
        if max_pages is not None and page_count[0] >= max_pages:
            print(f"[max_pages={max_pages} reached — stopping]")
            return

        indent   = "  " * depth
        page_num = f"[{page_count[0]+1}" + (f"/{max_pages}]" if max_pages else "]")
        print(f"\n{indent}Visiting {page_num}: {url}")

        visited.add(url)
        page_count[0] += 1

        result = PageResult(url=url, depth=depth, status_code=0)

        try:
            response           = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
            result.status_code = response.status_code

            # stage 2: content-type gate (uses real response, no extra request)
            if not filter_chain.allow_response(url, response):
                result.error = f"Content-Type blocked: {response.headers.get('Content-Type', '?')}"
                print(f"{indent}Skipped — {result.error}")
                results.append(result)
                return

            # ── parse — two soups, parsed once ────────────────────────────
            raw_soup = BeautifulSoup(response.text, "html.parser")
            soup     = BeautifulSoup(response.text, "html.parser")
            for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
                tag.decompose()

            # ── word count gate ────────────────────────────────────────────
            text = " ".join(soup.get_text(" ", strip=True).split())
            if word_count_threshold is not None:
                wc = len(text.split())
                if wc < word_count_threshold:
                    result.error = f"word_count={wc} < threshold={word_count_threshold}"
                    print(f"{indent}Skipped — {result.error}")
                    results.append(result)
                    return

            print(f"{indent}({len(text.split())} words): {text[:200]}...")

            # ── extract all requested outputs ──────────────────────────────
            manager.extract_all(result, soup, raw_soup, response)
            results.append(result)

            # ── collect next links ─────────────────────────────────────────
            clean_links = []
            seen_links  = set()

            for tag in raw_soup.find_all("a", href=True):
                clean_url = normalize_and_filter_url(url, tag["href"])
                if not clean_url:
                    continue
                if clean_url in seen_links or clean_url in visited:
                    continue
                if score_threshold is not None:
                    s = score_link(clean_url, tag.get_text())
                    if s < score_threshold:
                        print(f"{indent}  [score={s:.2f} < {score_threshold}] dropped: {clean_url}")
                        continue
                seen_links.add(clean_url)
                clean_links.append(clean_url)
                if branching_factor is not None and len(clean_links) >= branching_factor:
                    break

            for cl in clean_links:
                print(f"{indent}  -> {cl}")

            for cl in clean_links:
                _dfs_crawl(cl, depth + 1)

        except Exception as e:
            result.error = str(e)
            print(f"{indent}Error: {e}")
            results.append(result)

    # ── kick off ──────────────────────────────────────────────────────────────
    _dfs_crawl(start_url, depth=0)
    successful = len([r for r in results if not r.error])
    print(f"\nDone — {page_count[0]} visited, {successful} successful.")

    return results


# ── Usage ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    from outputs import MarkdownOutput, MetadataOutput, LinksOutput

    # # 1. crawl only
    # crawl("https://www.tensorflow.org/tutorials")

    # # 2. string names — quick and simple
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = ["markdown", "metadata", "links"],
    #     max_depth      = 1,
    #     max_pages      = 5,
    # )

    # # 3. pre-configured instances — full control over each extractor
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = [
    #         MarkdownOutput(heading_style="ATX", strip_links=True),
    #         MetadataOutput(),
    #         LinksOutput(include_external=False),
    #     ],
    #     max_depth      = 1,
    #     max_pages      = 5,
    # )

    # # 4. json — all formats, one serialisable object per page
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = ["json"],
    #     max_depth      = 1,
    #     max_pages      = 3,
    # )
    # for r in results:
    #     print(r.to_json())

    # 5. fully controlled
    results = crawl(
        start_url            = "https://www.tensorflow.org/tutorials",
        output_formats       = ["cleaned_html", "markdown", "metadata", "links"],
        max_depth            = 2,
        branching_factor     = 3,
        max_pages            = 3,
        score_threshold      = 0.3,
        word_count_threshold = 50,
        url_pattern          = "https://www.tensorflow.org/tutorials",
        exclude_external     = True,
        exclude_social_media = True,
        debug                = False,
    )
    for r in results:
        print(f"\n{'='*60}")
        print(f"URL:    {r.url}  |  depth={r.depth}  |  status={r.status_code}")
        if r.metadata:
            print(f"Title:  {r.metadata['title']}")
            print(f"Words:  {r.metadata['word_count']}")
        if r.links:
            print(f"Links:  {len(r.links)} found")
        if r.markdown:
            print(f"MD preview:\n{r.markdown[:400]}")


Visiting [1/3]: https://www.tensorflow.org/tutorials
(570 words): Tutorials | TensorFlow Core Skip to main content / English Español – América Latina Français Indonesia Italiano Polski Português – Brasil Tiếng Việt Türkçe Русский עברית العربيّة فارسی हिंदी বাংলা ภ...
  -> https://www.tensorflow.org/tutorials/quickstart/beginner
  -> https://www.tensorflow.org/tutorials/quickstart/advanced
  -> https://www.tensorflow.org/tutorials/keras/classification

  Visiting [2/3]: https://www.tensorflow.org/tutorials/quickstart/beginner
  (2448 words): TensorFlow 2 quickstart for beginners | TensorFlow Core Skip to main content / English Español – América Latina Français Indonesia Italiano Polski Português – Brasil Tiếng Việt Türkçe Русский עברית ...
    -> https://www.tensorflow.org/tutorials/quickstart/advanced
    -> https://www.tensorflow.org/tutorials/keras/classification
    -> https://www.tensorflow.org/tutorials/keras/text_classification

    Visiting [3/3]: https://www.tensorflow.org

# Multi-Output1

In [22]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib

import filters
importlib.reload(filters)
from filters import (FilterChain, DomainFilter, URLPatternFilter,
                     ContentTypeFilter, ExternalLinkFilter, SocialMediaFilter)

import outputs
importlib.reload(outputs)
from outputs import PageResult, OutputManager


def crawl(
    start_url,

    # ── output ─────────────────────────────────────────────────
    output_formats=None,        # list of str names or pre-configured BaseOutput instances
                                # e.g. ["markdown", "metadata"]
                                # e.g. [MarkdownOutput(strip_links=True), MetadataOutput()]
                                # e.g. ["json"]  → all formats, packed into to_json()
                                # None / []      → crawl only, no content stored

    # ── crawl controls ─────────────────────────────────────────
    max_depth=None,             # None = unlimited
    branching_factor=None,      # None = follow all links per page
    max_pages=None,             # None = no budget

    # ── content controls ───────────────────────────────────────
    score_threshold=None,       # None = off  |  e.g. 0.3
    word_count_threshold=None,  # None = off  |  e.g. 50

    # ── filter toggles ─────────────────────────────────────────
    url_pattern=None,           # None = off  |  e.g. "https://example.com/docs"
    exclude_external=False,
    exclude_social_media=False,
    extra_social_domains=None,

    # ── debug ──────────────────────────────────────────────────
    debug=False,
):
    """
    Single entry point for the DFS crawler.

    Parameters
    ----------
    start_url             : str
    output_formats        : list[str | BaseOutput]  (None = crawl only)
    max_depth             : int    (None = unlimited)
    branching_factor      : int    (None = all links)
    max_pages             : int    (None = unlimited)
    score_threshold       : float  (None = off)
    word_count_threshold  : int    (None = off)
    url_pattern           : str    (None = off)
    exclude_external      : bool
    exclude_social_media  : bool
    extra_social_domains  : list[str]
    debug                 : bool

    Returns
    -------
    list[PageResult]
    """

    # ── output manager ────────────────────────────────────────────────────────
    manager = OutputManager(output_formats or [])

    # ── filter chain ─────────────────────────────────────────────────────────
    active_filters = [DomainFilter(start_url)]

    if url_pattern is not None:
        active_filters.append(URLPatternFilter(url_pattern))
    if exclude_external:
        active_filters.append(ExternalLinkFilter(start_url))
    if exclude_social_media:
        active_filters.append(SocialMediaFilter(extra_domains=extra_social_domains))

    active_filters.append(ContentTypeFilter())

    filter_chain = FilterChain(active_filters, debug=debug)

    # ── crawl state ───────────────────────────────────────────────────────────
    visited    = set()
    page_count = [0]
    results    = []

    # ── helpers ───────────────────────────────────────────────────────────────

    def normalize_and_filter_url(current_url, href):
        absolute_url = urljoin(current_url, href)
        parsed       = urlparse(absolute_url)
        parsed       = parsed._replace(fragment="", query="")
        parsed       = parsed._replace(path=parsed.path.rstrip("/"))
        clean_url    = urlunparse(parsed)
        return clean_url if filter_chain.allow(clean_url) else None

    def score_link(href, anchor_text):
        score      = 0.0
        base_kw    = set(urlparse(start_url).path.strip("/").split("/")) - {""}
        link_parts = set(urlparse(href).path.strip("/").split("/"))      - {""}
        overlap    = len(base_kw & link_parts)
        score     += min(overlap / max(len(base_kw), 1), 1.0) * 0.5
        score     += 0.2 - min(len(link_parts) * 0.05, 0.2)
        if len(anchor_text.strip()) > 3:
            score += 0.3
        return round(min(score, 1.0), 3)

    # ── recursive worker ──────────────────────────────────────────────────────

    def _dfs_crawl(url, depth):

        url = url.rstrip("/")

        if max_depth is not None and depth > max_depth:
            return
        if url in visited:
            return
        if max_pages is not None and page_count[0] >= max_pages:
            print(f"[max_pages={max_pages} reached — stopping]")
            return

        indent   = "  " * depth
        page_num = f"[{page_count[0]+1}" + (f"/{max_pages}]" if max_pages else "]")
        print(f"\n{indent}Visiting {page_num}: {url}")

        visited.add(url)
        page_count[0] += 1

        result = PageResult(url=url, depth=depth, status_code=0)

        try:
            response           = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
            result.status_code = response.status_code

            # stage 2: content-type gate (uses real response, no extra request)
            if not filter_chain.allow_response(url, response):
                result.error = f"Content-Type blocked: {response.headers.get('Content-Type', '?')}"
                print(f"{indent}Skipped — {result.error}")
                results.append(result)
                return

            # ── parse — two soups, parsed once ────────────────────────────
            raw_soup = BeautifulSoup(response.text, "html.parser")
            soup     = BeautifulSoup(response.text, "html.parser")

            # Pass 1: standard semantic noise tags
            NOISE_TAGS = [
                "script", "style",
                "nav", "header", "footer", "aside",
                "form", "iframe", "noscript",
            ]
            for tag in soup(NOISE_TAGS):
                tag.decompose()

            # Pass 2: custom elements and class/id patterns
            # Catches web components like <devsite-header>, <site-nav>
            # and class names like "cookie-banner", "sidebar", "toc"
            #
            # IMPORTANT: snapshot with list() first — decomposing a parent
            # also destroys its children in the tree, so iterating live
            # causes tag.name / tag.get() to return None on already-dead nodes.
            NOISE_PATTERNS = [
                "header", "footer", "nav", "cookie", "banner",
                "breadcrumb", "sidebar", "toc", "toolbar", "menu",
                "announcement", "notification", "skip", "search",
            ]
            for tag in list(soup.find_all(True)):
                if tag.parent is None:      # already decomposed as child of earlier tag
                    continue
                combined = " ".join([
                    tag.name or "",
                    " ".join(tag.get("class") or []),
                    tag.get("id") or "",
                ]).lower()
                if any(p in combined for p in NOISE_PATTERNS):
                    tag.decompose()

            # Content scoping: prefer the most specific content container
            # Priority: <main> → <article> → role=main → <body> → full soup
            content = (
                soup.find("main")
                or soup.find("article")
                or soup.find(attrs={"role": "main"})
                or soup.find("body")
                or soup
            )

            # ── word count gate ────────────────────────────────────────────
            text = " ".join(content.get_text(" ", strip=True).split())
            if word_count_threshold is not None:
                wc = len(text.split())
                if wc < word_count_threshold:
                    result.error = f"word_count={wc} < threshold={word_count_threshold}"
                    print(f"{indent}Skipped — {result.error}")
                    results.append(result)
                    return

            print(f"{indent}({len(text.split())} words): {text[:200]}...")

            # ── extract all requested outputs ──────────────────────────────
            # pass content (scoped) as the clean soup so all extractors
            # that use needs_clean=True operate on main-content only
            manager.extract_all(result, content, raw_soup, response)
            results.append(result)

            # ── collect next links ─────────────────────────────────────────
            clean_links = []
            seen_links  = set()

            for tag in raw_soup.find_all("a", href=True):
                clean_url = normalize_and_filter_url(url, tag["href"])
                if not clean_url:
                    continue
                if clean_url in seen_links or clean_url in visited:
                    continue
                if score_threshold is not None:
                    s = score_link(clean_url, tag.get_text())
                    if s < score_threshold:
                        print(f"{indent}  [score={s:.2f} < {score_threshold}] dropped: {clean_url}")
                        continue
                seen_links.add(clean_url)
                clean_links.append(clean_url)
                if branching_factor is not None and len(clean_links) >= branching_factor:
                    break

            for cl in clean_links:
                print(f"{indent}  -> {cl}")

            for cl in clean_links:
                _dfs_crawl(cl, depth + 1)

        except Exception as e:
            result.error = str(e)
            print(f"{indent}Error: {e}")
            results.append(result)

    # ── kick off ──────────────────────────────────────────────────────────────
    _dfs_crawl(start_url, depth=0)
    successful = len([r for r in results if not r.error])
    print(f"\nDone — {page_count[0]} visited, {successful} successful.")

    return results


# ── Usage ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    from outputs import MarkdownOutput, MetadataOutput, LinksOutput

    # # 1. crawl only
    # crawl("https://www.tensorflow.org/tutorials")

    # # 2. string names — quick and simple
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = ["markdown", "metadata", "links"],
    #     max_depth      = 1,
    #     max_pages      = 5,
    # )

    # # 3. pre-configured instances — full control over each extractor
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = [
    #         MarkdownOutput(heading_style="ATX", strip_links=True),
    #         MetadataOutput(),
    #         LinksOutput(include_external=False),
    #     ],
    #     max_depth      = 1,
    #     max_pages      = 5,
    # )

    # # 4. json — all formats, one serialisable object per page
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = ["json"],
    #     max_depth      = 1,
    #     max_pages      = 3,
    # )
    for r in results:
        print(r.to_json())

    # 5. fully controlled
    results = crawl(
        start_url            = "https://www.tensorflow.org/tutorials",
        output_formats       = ["cleaned_html", "markdown", "metadata", "links"],
        max_depth            = 2,
        branching_factor     = 3,
        max_pages            = 10,
        score_threshold      = 0.3,
        word_count_threshold = 50,
        url_pattern          = "https://www.tensorflow.org/tutorials",
        exclude_external     = True,
        exclude_social_media = True,
        debug                = False,
    )
    for r in results:
        print(f"\n{'='*60}")
        print(f"URL:    {r.url}  |  depth={r.depth}  |  status={r.status_code}")
        if r.metadata:
            print(f"Title:  {r.metadata['title']}")
            print(f"Words:  {r.metadata['word_count']}")
        if r.links:
            print(f"Links:  {len(r.links)} found")
        if r.markdown:
            print(f"MD preview:\n{r.markdown[:400]}")

{
  "url": "https://www.tensorflow.org/tutorials",
  "depth": 0,
  "status_code": 200,
  "error": "'NoneType' object has no attribute 'get'",
  "cleaned_html": null,
  "raw_html": null,
  "markdown": null,
  "metadata": null,
  "links": null
}

Visiting [1/10]: https://www.tensorflow.org/tutorials
(449 words): Stay organized with collections Save and categorize content based on your preferences. The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook envi...
  -> https://www.tensorflow.org/tutorials/quickstart/beginner
  -> https://www.tensorflow.org/tutorials/quickstart/advanced
  -> https://www.tensorflow.org/tutorials/keras/classification

  Visiting [2/10]: https://www.tensorflow.org/tutorials/quickstart/beginner
  (2346 words): TensorFlow 2 quickstart for beginners Stay organized with collections Save and categorize content based on your preferences. View on TensorFlow.org Run in Google Colab View source on GitHub Download n...


# Multi-Output2

In [26]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urlunparse
import importlib

import filters
importlib.reload(filters)
from filters import (FilterChain, DomainFilter, URLPatternFilter,
                     ContentTypeFilter, ExternalLinkFilter, SocialMediaFilter)

import outputs
importlib.reload(outputs)
from outputs import PageResult, OutputManager


def crawl(
    start_url,

    # ── output ─────────────────────────────────────────────────
    output_formats=None,        # list of str names or pre-configured BaseOutput instances
                                # e.g. ["markdown", "metadata"]
                                # e.g. [MarkdownOutput(strip_links=True), MetadataOutput()]
                                # e.g. ["json"]  → all formats, packed into to_json()
                                # None / []      → crawl only, no content stored

    # ── crawl controls ─────────────────────────────────────────
    max_depth=None,             # None = unlimited
    branching_factor=None,      # None = follow all links per page
    max_pages=None,             # None = no budget

    # ── content controls ───────────────────────────────────────
    score_threshold=None,       # None = off  |  e.g. 0.3
    word_count_threshold=None,  # None = off  |  e.g. 50

    # ── filter toggles ─────────────────────────────────────────
    url_pattern=None,           # None = off  |  e.g. "https://example.com/docs"
    exclude_external=False,
    exclude_social_media=False,
    extra_social_domains=None,

    # ── debug ──────────────────────────────────────────────────
    debug=False,
):
    """
    Single entry point for the DFS crawler.

    Parameters
    ----------
    start_url             : str
    output_formats        : list[str | BaseOutput]  (None = crawl only)
    max_depth             : int    (None = unlimited)
    branching_factor      : int    (None = all links)
    max_pages             : int    (None = unlimited)
    score_threshold       : float  (None = off)
    word_count_threshold  : int    (None = off)
    url_pattern           : str    (None = off)
    exclude_external      : bool
    exclude_social_media  : bool
    extra_social_domains  : list[str]
    debug                 : bool

    Returns
    -------
    list[PageResult]
    """

    # ── output manager ────────────────────────────────────────────────────────
    manager = OutputManager(output_formats or [])

    # ── filter chain ─────────────────────────────────────────────────────────
    active_filters = [DomainFilter(start_url)]

    if url_pattern is not None:
        active_filters.append(URLPatternFilter(url_pattern))
    if exclude_external:
        active_filters.append(ExternalLinkFilter(start_url))
    if exclude_social_media:
        active_filters.append(SocialMediaFilter(extra_domains=extra_social_domains))

    active_filters.append(ContentTypeFilter())

    filter_chain = FilterChain(active_filters, debug=debug)

    # ── crawl state ───────────────────────────────────────────────────────────
    visited    = set()
    page_count = [0]
    results    = []

    # ── helpers ───────────────────────────────────────────────────────────────

    def normalize_and_filter_url(current_url, href):
        absolute_url = urljoin(current_url, href)
        parsed       = urlparse(absolute_url)
        parsed       = parsed._replace(fragment="", query="")
        parsed       = parsed._replace(path=parsed.path.rstrip("/"))
        clean_url    = urlunparse(parsed)
        return clean_url if filter_chain.allow(clean_url) else None

    def score_link(href, anchor_text):
        score      = 0.0
        base_kw    = set(urlparse(start_url).path.strip("/").split("/")) - {""}
        link_parts = set(urlparse(href).path.strip("/").split("/"))      - {""}
        overlap    = len(base_kw & link_parts)
        score     += min(overlap / max(len(base_kw), 1), 1.0) * 0.5
        score     += 0.2 - min(len(link_parts) * 0.05, 0.2)
        if len(anchor_text.strip()) > 3:
            score += 0.3
        return round(min(score, 1.0), 3)

    # ── recursive worker ──────────────────────────────────────────────────────

    def _dfs_crawl(url, depth):

        url = url.rstrip("/")

        if max_depth is not None and depth > max_depth:
            return
        if url in visited:
            return
        if max_pages is not None and page_count[0] >= max_pages:
            print(f"[max_pages={max_pages} reached — stopping]")
            return

        indent   = "  " * depth
        page_num = f"[{page_count[0]+1}" + (f"/{max_pages}]" if max_pages else "]")
        print(f"\n{indent}Visiting {page_num}: {url}")

        visited.add(url)
        page_count[0] += 1

        result = PageResult(url=url, depth=depth, status_code=0)

        try:
            response           = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=5)
            result.status_code = response.status_code

            # stage 2: content-type gate (uses real response, no extra request)
            if not filter_chain.allow_response(url, response):
                result.error = f"Content-Type blocked: {response.headers.get('Content-Type', '?')}"
                print(f"{indent}Skipped — {result.error}")
                results.append(result)
                return

            # ── parse — two soups, parsed once ────────────────────────────
            raw_soup = BeautifulSoup(response.text, "html.parser")
            soup     = BeautifulSoup(response.text, "html.parser")

            # Pass 1: standard semantic noise tags
            NOISE_TAGS = [
                "script", "style",
                "nav", "header", "footer", "aside",
                "form", "iframe", "noscript",
            ]
            for tag in soup(NOISE_TAGS):
                tag.decompose()

            # Pass 2: custom elements and class/id patterns
            # Catches web components like <devsite-header>, <site-nav>
            # and class names like "cookie-banner", "sidebar", "toc"
            #
            # IMPORTANT: snapshot with list() first — decomposing a parent
            # also destroys its children in the tree, so iterating live
            # causes tag.name / tag.get() to return None on already-dead nodes.
            NOISE_PATTERNS = [
                "header", "footer", "nav", "cookie", "banner",
                "breadcrumb", "sidebar", "toc", "toolbar", "menu",
                "announcement", "notification", "skip", "search",
                "devsite-band", "devsite-collection", "devsite-rating",
                "devsite-thumb", "devsite-page-rating", "devsite-bookmark",
            ]
            for tag in list(soup.find_all(True)):
                if tag.parent is None:      # already decomposed as child of earlier tag
                    continue
                combined = " ".join([
                    tag.name or "",
                    " ".join(tag.get("class") or []),
                    tag.get("id") or "",
                ]).lower()
                if any(p in combined for p in NOISE_PATTERNS):
                    tag.decompose()

            # Content scoping: prefer the most specific content container
            # Priority: <main> → <article> → role=main → <body> → full soup
            content = (
                soup.find("main")
                or soup.find("article")
                or soup.find(attrs={"role": "main"})
                or soup.find("body")
                or soup
            )

            # ── word count gate ────────────────────────────────────────────
            text = " ".join(content.get_text(" ", strip=True).split())
            if word_count_threshold is not None:
                wc = len(text.split())
                if wc < word_count_threshold:
                    result.error = f"word_count={wc} < threshold={word_count_threshold}"
                    print(f"{indent}Skipped — {result.error}")
                    results.append(result)
                    return

            print(f"{indent}({len(text.split())} words): {text[:200]}...")

            # ── extract all requested outputs ──────────────────────────────
            # pass content (scoped) as the clean soup so all extractors
            # that use needs_clean=True operate on main-content only
            manager.extract_all(result, content, raw_soup, response)
            results.append(result)

            # ── collect next links ─────────────────────────────────────────
            clean_links = []
            seen_links  = set()

            for tag in raw_soup.find_all("a", href=True):
                clean_url = normalize_and_filter_url(url, tag["href"])
                if not clean_url:
                    continue
                if clean_url in seen_links or clean_url in visited:
                    continue
                if score_threshold is not None:
                    s = score_link(clean_url, tag.get_text())
                    if s < score_threshold:
                        print(f"{indent}  [score={s:.2f} < {score_threshold}] dropped: {clean_url}")
                        continue
                seen_links.add(clean_url)
                clean_links.append(clean_url)
                if branching_factor is not None and len(clean_links) >= branching_factor:
                    break

            for cl in clean_links:
                print(f"{indent}  -> {cl}")

            for cl in clean_links:
                _dfs_crawl(cl, depth + 1)

        except Exception as e:
            result.error = str(e)
            print(f"{indent}Error: {e}")
            results.append(result)

    # ── kick off ──────────────────────────────────────────────────────────────
    _dfs_crawl(start_url, depth=0)
    successful = len([r for r in results if not r.error])
    print(f"\nDone — {page_count[0]} visited, {successful} successful.")

    return results


# ── Usage ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    from outputs import MarkdownOutput, MetadataOutput, LinksOutput

    # # 1. crawl only
    # crawl("https://www.tensorflow.org/tutorials")

    # # 2. string names — quick and simple
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = ["markdown", "metadata", "links"],
    #     max_depth      = 1,
    #     max_pages      = 5,
    # )

    # # 3. pre-configured instances — full control over each extractor
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = [
    #         MarkdownOutput(heading_style="ATX", strip_links=True),
    #         MetadataOutput(),
    #         LinksOutput(include_external=False),
    #     ],
    #     max_depth      = 1,
    #     max_pages      = 5,
    # )

    # # 4. json — all formats, one serialisable object per page
    # results = crawl(
    #     start_url      = "https://www.tensorflow.org/tutorials",
    #     output_formats = ["json"],
    #     max_depth      = 1,
    #     max_pages      = 3,
    # )
    # for r in results:
    #     print(r.to_json())

    # 5. fully controlled
    results = crawl(
        start_url            = "https://www.tensorflow.org/tutorials",
        output_formats       = ["cleaned_html", "markdown", "metadata", "links"],
        max_depth            = 2,
        branching_factor     = 3,
        max_pages            = 10,
        score_threshold      = 0.3,
        word_count_threshold = 50,
        url_pattern          = "https://www.tensorflow.org/tutorials",
        exclude_external     = True,
        exclude_social_media = True,
        debug                = False,
    )
    for r in results:
        print(f"\n{'='*60}")
        print(f"URL:    {r.url}  |  depth={r.depth}  |  status={r.status_code}")
        if r.metadata:
            print(f"Title:  {r.metadata['title']}")
            print(f"Words:  {r.metadata['word_count']}")
        if r.links:
            print(f"Links:  {len(r.links)} found")
        if r.markdown:
            print(f"MD preview:\n{r.markdown[:400]}")


Visiting [1/10]: https://www.tensorflow.org/tutorials
(437 words): The TensorFlow tutorials are written as Jupyter notebooks and run directly in Google Colab—a hosted notebook environment that requires no setup. At the top of each tutorial, you'll see a Run in Google...
  -> https://www.tensorflow.org/tutorials/quickstart/beginner
  -> https://www.tensorflow.org/tutorials/quickstart/advanced
  -> https://www.tensorflow.org/tutorials/keras/classification

  Visiting [2/10]: https://www.tensorflow.org/tutorials/quickstart/beginner
  (2334 words): TensorFlow 2 quickstart for beginners View on TensorFlow.org Run in Google Colab View source on GitHub Download notebook This short introduction uses Keras to: Load a prebuilt dataset. Build a neural ...
    -> https://www.tensorflow.org/tutorials/quickstart/advanced
    -> https://www.tensorflow.org/tutorials/keras/classification
    -> https://www.tensorflow.org/tutorials/keras/text_classification

    Visiting [3/10]: https://www.tensorflow.